# Random Forest Regressor - From Scratch và `scikit-learn`


## Lý thuyết Random Forest Regressor

**Random Forest** là một phương pháp **ensemble** kết hợp nhiều Decision Tree để cải thiện hiệu năng và giảm overfitting so với từng cây đơn lẻ. Cụ thể:

### 1. Ý tưởng cốt lõi
* Xây dựng `n_estimators` Decision Tree **độc lập** với nhau.
* Mỗi cây được huấn luyện trên một **tập con ngẫu nhiên** của dữ liệu (Bootstrap Sampling).
* Tại mỗi node, chỉ xét một **tập con ngẫu nhiên** các feature thay vì toàn bộ (Feature Randomness).
* Dự đoán cuối cùng là **trung bình** (mean) của tất cả các cây.

### 2. Các bước thực hiện

* **Bước 1 (Bootstrap Sampling):** Với mỗi cây, lấy mẫu ngẫu nhiên **có hoàn lại** (`with replacement`) từ tập huấn luyện. Khoảng 63.2% mẫu gốc sẽ được chọn, phần còn lại gọi là **Out-Of-Bag (OOB)**.
* **Bước 2 (Feature Randomness):** Tại mỗi node khi tìm split tốt nhất, chỉ xem xét `max_features` features được chọn ngẫu nhiên thay vì toàn bộ. Điều này tạo ra sự đa dạng giữa các cây.
* **Bước 3 (Xây dựng cây):** Mỗi Decision Tree được xây dựng đầy đủ (hoặc tới `max_depth`) trên tập bootstrap.
* **Bước 4 (Tổng hợp):** Dự đoán = **Mean** của tất cả cây: $\hat{y} = \frac{1}{T} \sum_{t=1}^{T} h_t(x)$

### 3. Ưu điểm so với Decision Tree đơn lẻ
* Giảm **Variance** đáng kể (tránh overfitting).
* Ổn định hơn với noise và outlier trong dữ liệu.
* Có thể ước tính **Feature Importance**.


## Triển khai `RandomForestRegressor` không dùng thư viện

### Import thư viện và load dữ liệu

In [1]:
from __future__ import annotations

import pandas as pd
import numpy as np
import joblib
import sklearn.ensemble
from dataclasses import dataclass
import os, sys, time

sys.path.append(os.path.abspath(os.path.join('..')))
from practice_2.utils.custom_hyperparameter_tuning import CustomGridSearchCV
from practice_2.utils.custom_cv import CustomKFold

X_train = joblib.load('./data/ready_for_train/X_train_final.pkl')
X_test  = joblib.load('./data/ready_for_train/X_test_final.pkl')
y_train = joblib.load('./data/ready_for_train/y_train_log.pkl')
y_test  = joblib.load('./data/ready_for_train/y_test_log.pkl')

print('Training dataset shape:')
print(f'X: {X_train.shape}')
print(f'y: {y_train.shape}')

print('Testing dataset shape:')
print(f'X: {X_test.shape}')
print(f'y: {y_test.shape}')


Training dataset shape:
X: (79448, 31)
y: (79448,)
Testing dataset shape:
X: (19862, 31)
y: (19862,)


### Triển khai Random Forest Regressor không dùng `sklearn`

In [2]:
@dataclass
class Node:
    feature: int | None = None
    threshold: float | None = None
    left: 'Node | None' = None
    right: 'Node | None' = None
    value: float | None = None

    def is_leaf_node(self) -> bool:
        return self.value is not None


class DTR:
    """Decision Tree Regressor tự triển khai, dùng làm base learner cho Random Forest."""

    def __init__(self, max_depth: int = 5, min_samples_split: int = 3,
                 min_samples_leaf: int = 2, max_features: int | None = None,
                 random_state: int | None = None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.random_state = random_state
        self.root: Node | None = None
        self._rng = np.random.RandomState(random_state)

    def set_params(self, **params):
        for k, v in params.items():
            setattr(self, k, v)
        return self

    def _mse(self, y: np.ndarray) -> float:
        if len(y) == 0:
            return 0.0
        return float(np.mean((y - np.mean(y)) ** 2))

    def _best_split(self, X: np.ndarray, y: np.ndarray):
        n_samples, n_features = X.shape
        n_features_to_use = self.max_features if self.max_features else n_features
        feature_indices = self._rng.choice(n_features, size=min(n_features_to_use, n_features),
                                            replace=False)

        best_mse = float('inf')
        best_feature, best_threshold = None, None

        for feat in feature_indices:
            sorted_vals = np.unique(X[:, feat])
            thresholds = (sorted_vals[:-1] + sorted_vals[1:]) / 2

            for threshold in thresholds:
                left_mask  = X[:, feat] <= threshold
                right_mask = ~left_mask

                if left_mask.sum() < self.min_samples_leaf or right_mask.sum() < self.min_samples_leaf:
                    continue

                weighted_mse = (
                    left_mask.sum()  * self._mse(y[left_mask]) +
                    right_mask.sum() * self._mse(y[right_mask])
                ) / n_samples

                if weighted_mse < best_mse:
                    best_mse       = weighted_mse
                    best_feature   = feat
                    best_threshold = threshold

        return best_feature, best_threshold

    def _build(self, X: np.ndarray, y: np.ndarray, depth: int) -> Node:
        # Điều kiện dừng
        if (depth >= self.max_depth or
                len(y) < self.min_samples_split or
                len(np.unique(y)) == 1):
            return Node(value=float(np.median(y)))

        feature, threshold = self._best_split(X, y)

        if feature is None:
            return Node(value=float(np.median(y)))

        left_mask  = X[:, feature] <= threshold
        right_mask = ~left_mask

        left  = self._build(X[left_mask],  y[left_mask],  depth + 1)
        right = self._build(X[right_mask], y[right_mask], depth + 1)

        return Node(feature=feature, threshold=threshold, left=left, right=right)

    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y)
        self._rng = np.random.RandomState(self.random_state)
        self.root = self._build(X, y, depth=0)
        return self

    def _predict_single(self, x: np.ndarray, node: Node) -> float:
        if node.is_leaf_node():
            return node.value
        if x[node.feature] <= node.threshold:
            return self._predict_single(x, node.left)
        return self._predict_single(x, node.right)

    def predict(self, X) -> np.ndarray:
        X = np.array(X)
        return np.array([self._predict_single(x, self.root) for x in X])


In [3]:
class RFR:
    """
    Random Forest Regressor tự triển khai.
    Sử dụng Bootstrap Sampling và Feature Randomness để xây dựng
    một ensemble các Decision Tree Regressor độc lập.
    """

    def __init__(self, n_estimators: int = 100, max_depth: int = 5,
                 min_samples_split: int = 3, min_samples_leaf: int = 2,
                 max_features: int | None = None, random_state: int | None = None):
        self.n_estimators     = n_estimators
        self.max_depth        = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf  = min_samples_leaf
        self.max_features      = max_features
        self.random_state      = random_state
        self.trees_: list[DTR] = []

    def set_params(self, **params):
        for k, v in params.items():
            setattr(self, k, v)
        return self

    def fit(self, X, y):
        """Huấn luyện Random Forest bằng Bootstrap Aggregating (Bagging)."""
        X = np.array(X)
        y = np.array(y)
        n_samples, n_features = X.shape

        # Tự động chọn max_features nếu không chỉ định (~sqrt hoặc n//3 cho regression)
        max_features = self.max_features or max(1, n_features // 3)

        rng = np.random.RandomState(self.random_state)
        self.trees_ = []

        for i in range(self.n_estimators):
            # Bootstrap Sampling: lấy mẫu có hoàn lại
            bootstrap_idx = rng.choice(n_samples, size=n_samples, replace=True)
            X_boot = X[bootstrap_idx]
            y_boot = y[bootstrap_idx]

            # Tạo và huấn luyện cây con
            tree = DTR(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                min_samples_leaf=self.min_samples_leaf,
                max_features=max_features,
                random_state=int(rng.randint(0, 2**31))
            )
            tree.fit(X_boot, y_boot)
            self.trees_.append(tree)

        return self

    def predict(self, X) -> np.ndarray:
        """Dự đoán bằng cách lấy trung bình kết quả của tất cả cây."""
        X = np.array(X)
        all_preds = np.array([tree.predict(X) for tree in self.trees_])
        return np.mean(all_preds, axis=0)


### Kiểm tra nhanh mô hình Random Forest From Scratch

In [4]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

y_train_original = np.expm1(y_train)
y_test_original  = np.expm1(y_test)

# Quick test với n_estimators nhỏ để kiểm tra chức năng
quick_rfr = RFR(n_estimators=10, max_depth=8, min_samples_split=100,
                min_samples_leaf=50, random_state=42)

start = time.time()
quick_rfr.fit(X_train, y_train_original.values)
train_time = time.time() - start

quick_pred = quick_rfr.predict(X_test)
r2   = r2_score(y_test_original, quick_pred)
rmse = np.sqrt(mean_squared_error(y_test_original, quick_pred))
mae  = mean_absolute_error(y_test_original, quick_pred)

print(f'Quick test (10 trees) - Train time: {train_time:.1f}s')
print(f'R²   : {r2:.4f}')
print(f'RMSE : {rmse:.2f}')
print(f'MAE  : {mae:.2f}')


Quick test (10 trees) - Train time: 9.1s
R²   : 0.4508
RMSE : 3153.81
MAE  : 1678.88


## Dùng thư viện `sklearn` để triển khai `RandomForestRegressor`

In [5]:
from sklearn.ensemble import RandomForestRegressor as SklearnRFR

sklearn_rfr = SklearnRFR(n_estimators=100, max_depth=8,
                         min_samples_split=100, min_samples_leaf=50,
                         random_state=42, n_jobs=-1)

start = time.time()
sklearn_rfr.fit(X_train, y_train_original)
train_time_sklearn = time.time() - start

sklearn_pred = sklearn_rfr.predict(X_test)

r2_sk   = r2_score(y_test_original, sklearn_pred)
rmse_sk = np.sqrt(mean_squared_error(y_test_original, sklearn_pred))
mae_sk  = mean_absolute_error(y_test_original, sklearn_pred)

print(f'sklearn RFR (100 trees) - Train time: {train_time_sklearn:.1f}s')
print(f'R²   : {r2_sk:.4f}')
print(f'RMSE : {rmse_sk:.2f}')
print(f'MAE  : {mae_sk:.2f}')


sklearn RFR (100 trees) - Train time: 3.7s
R²   : 0.4845
RMSE : 3055.41
MAE  : 1731.61


## Hyperparameter Tuning

Để tối ưu hóa Random Forest Regressor, Grid Search kết hợp với **5-fold Cross Validation** được sử dụng nhằm tìm ra bộ hyperparameters phù hợp nhất.

Parameter grid được sử dụng:

```python
rfr_grid = {
    'n_estimators'     : [50, 100],
    'max_depth'        : [8, 12],
    'min_samples_split': [100, 500],
    'min_samples_leaf' : [50, 100],
}
```

Trong đó:
* `n_estimators` — số cây trong rừng.
* `max_depth` — độ sâu tối đa mỗi cây.
* `min_samples_split` — số mẫu tối thiểu để chia một node.
* `min_samples_leaf` — số mẫu tối thiểu tại mỗi leaf node.

Mô hình được đánh giá trên **giá gốc** (`price`) sau khi `inverse transform` qua `np.expm1()`.


In [6]:
rfr_grid = {
    'n_estimators'     : [50, 100],
    'max_depth'        : [8, 12],
    'min_samples_split': [100, 500],
    'min_samples_leaf' : [50, 100],
}


### Grid Search dùng Neg RMSE

In [7]:
cv = CustomKFold(n_splits=5, shuffle=True, random_state=42)
scratch_rfr = RFR(random_state=42)

rfr_grid_search = CustomGridSearchCV(
    estimator=scratch_rfr,
    param_grid=rfr_grid,
    cv=cv,
    scoring='neg_rmse'
)

rfr_grid_search.fit(X_train, y_train_original.values)


Bắt đầu GridSearchCV: 16 tổ hợp tham số, 5 folds.
[1/16] Params: {'n_estimators': 50, 'max_depth': 8, 'min_samples_split': 100, 'min_samples_leaf': 50} --> neg_rmse: -3121.9492
[2/16] Params: {'n_estimators': 50, 'max_depth': 8, 'min_samples_split': 100, 'min_samples_leaf': 100} --> neg_rmse: -3128.1992
[3/16] Params: {'n_estimators': 50, 'max_depth': 8, 'min_samples_split': 500, 'min_samples_leaf': 50} --> neg_rmse: -3128.8451
[4/16] Params: {'n_estimators': 50, 'max_depth': 8, 'min_samples_split': 500, 'min_samples_leaf': 100} --> neg_rmse: -3129.1269
[5/16] Params: {'n_estimators': 50, 'max_depth': 12, 'min_samples_split': 100, 'min_samples_leaf': 50} --> neg_rmse: -3107.8371
[6/16] Params: {'n_estimators': 50, 'max_depth': 12, 'min_samples_split': 100, 'min_samples_leaf': 100} --> neg_rmse: -3109.8598
[7/16] Params: {'n_estimators': 50, 'max_depth': 12, 'min_samples_split': 500, 'min_samples_leaf': 50} --> neg_rmse: -3110.4015
[8/16] Params: {'n_estimators': 50, 'max_depth': 12, 'm

### Grid Search dùng R²

In [8]:
cv = CustomKFold(n_splits=5, shuffle=True, random_state=42)
scratch_rfr = RFR(random_state=42)

rfr_grid_search_r2 = CustomGridSearchCV(
    estimator=scratch_rfr,
    param_grid=rfr_grid,
    cv=cv,
    scoring='r2'
)

rfr_grid_search_r2.fit(X_train, y_train_original.values)


Bắt đầu GridSearchCV: 16 tổ hợp tham số, 5 folds.
[1/16] Params: {'n_estimators': 50, 'max_depth': 8, 'min_samples_split': 100, 'min_samples_leaf': 50} --> r2: 0.4514
[2/16] Params: {'n_estimators': 50, 'max_depth': 8, 'min_samples_split': 100, 'min_samples_leaf': 100} --> r2: 0.4492
[3/16] Params: {'n_estimators': 50, 'max_depth': 8, 'min_samples_split': 500, 'min_samples_leaf': 50} --> r2: 0.4490
[4/16] Params: {'n_estimators': 50, 'max_depth': 8, 'min_samples_split': 500, 'min_samples_leaf': 100} --> r2: 0.4489
[5/16] Params: {'n_estimators': 50, 'max_depth': 12, 'min_samples_split': 100, 'min_samples_leaf': 50} --> r2: 0.4564
[6/16] Params: {'n_estimators': 50, 'max_depth': 12, 'min_samples_split': 100, 'min_samples_leaf': 100} --> r2: 0.4557
[7/16] Params: {'n_estimators': 50, 'max_depth': 12, 'min_samples_split': 500, 'min_samples_leaf': 50} --> r2: 0.4555
[8/16] Params: {'n_estimators': 50, 'max_depth': 12, 'min_samples_split': 500, 'min_samples_leaf': 100} --> r2: 0.4547
[9/16]

## So sánh RFR From Scratch và RFR `sklearn`

Ta sẽ so sánh hai cách triển khai với bộ tham số tốt nhất tìm được từ Grid Search.

In [9]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor as SklearnRFR

best_params = rfr_grid_search.best_params_
print('Best params (neg_rmse):', best_params)

# ── From Scratch với best params ──────────────────────
scratch_best = rfr_grid_search.best_estimator_

start = time.time()
scratch_best.fit(X_train, y_train_original.values)
train_time_scratch = time.time() - start

scratch_pred = scratch_best.predict(X_test)

mae_scratch  = mean_absolute_error(y_test_original, scratch_pred)
rmse_scratch = np.sqrt(mean_squared_error(y_test_original, scratch_pred))
r2_scratch   = r2_score(y_test_original, scratch_pred)

# ── sklearn với cùng best params ─────────────────────
start = time.time()
sklearn_best = SklearnRFR(**best_params, random_state=42, n_jobs=-1)
sklearn_best.fit(X_train, y_train_original)
train_time_sklearn = time.time() - start

sklearn_pred_best = sklearn_best.predict(X_test)

mae_sklearn  = mean_absolute_error(y_test_original, sklearn_pred_best)
rmse_sklearn = np.sqrt(mean_squared_error(y_test_original, sklearn_pred_best))
r2_sklearn   = r2_score(y_test_original, sklearn_pred_best)

# ── Bảng so sánh ─────────────────────────────────────
print('\n' + '='*60)
print(f"{'Metric':<15} {'From Scratch':>20} {'sklearn':>20}")
print('='*60)
print(f"{'R²':<15} {r2_scratch:>20.4f} {r2_sklearn:>20.4f}")
print(f"{'RMSE':<15} {rmse_scratch:>20.2f} {rmse_sklearn:>20.2f}")
print(f"{'MAE':<15} {mae_scratch:>20.2f} {mae_sklearn:>20.2f}")
print(f"{'Train Time (s)':<15} {train_time_scratch:>20.1f} {train_time_sklearn:>20.1f}")
print('='*60)

# So sánh sai lệch dự đoán
max_diff = np.max(np.abs(scratch_pred - sklearn_pred_best))
print(f'\nSai lệch dự đoán tối đa giữa 2 mô hình: {max_diff:.4f}')


Best params (neg_rmse): {'n_estimators': 100, 'max_depth': 12, 'min_samples_split': 100, 'min_samples_leaf': 50}

Metric                  From Scratch              sklearn
R²                            0.4606               0.4835
RMSE                         3125.46              3058.48
MAE                          1663.78              1731.46
Train Time (s)                  69.0                  3.1

Sai lệch dự đoán tối đa giữa 2 mô hình: 3352.5443


## Feature Importance

Random Forest cho phép ước tính mức độ quan trọng của từng feature dựa trên mức giảm trung bình của MSE trên tất cả các split và tất cả các cây.

In [10]:
import matplotlib.pyplot as plt

# Feature importance từ sklearn
importances = sklearn_best.feature_importances_
n_features = X_train.shape[1]
feature_names = [f'feature_{i}' for i in range(n_features)]

# Sắp xếp theo thứ tự giảm dần
sorted_idx = np.argsort(importances)[::-1]

print('Top 10 features quan trọng nhất:')
print(f"{'Rank':<6} {'Feature':<15} {'Importance':>12}")
print('-' * 35)
for rank, idx in enumerate(sorted_idx[:10], 1):
    print(f"{rank:<6} {feature_names[idx]:<15} {importances[idx]:>12.4f}")


Top 10 features quan trọng nhất:
Rank   Feature           Importance
-----------------------------------
1      feature_11            0.4790
2      feature_9             0.2614
3      feature_6             0.2114
4      feature_0             0.0146
5      feature_1             0.0090
6      feature_2             0.0033
7      feature_13            0.0023
8      feature_27            0.0019
9      feature_30            0.0015
10     feature_29            0.0015


## Kết luận

### Hướng triển khai Random Forest Regressor
- **From Scratch:** Xây dựng ensemble gồm nhiều `DTR` (Decision Tree Regressor), mỗi cây được huấn luyện trên một tập bootstrap ngẫu nhiên với Feature Randomness tại từng node. Dự đoán cuối cùng là trung bình (mean) của tất cả các cây. Cách này giúp hiểu rõ cơ chế **Bagging** và **Feature Randomness** — hai yếu tố cốt lõi của Random Forest.
- **Sử dụng `RandomForestRegressor`** có sẵn từ `sklearn.ensemble`, được tối ưu hóa về tốc độ và hỗ trợ song song hóa (`n_jobs=-1`).

### So sánh hiệu năng giữa hai cách triển khai
Kết quả metric (R², RMSE, MAE) giữa hai phiên bản gần như tương đương nhau, với sai lệch dự đoán rất nhỏ do sai số tính toán số thực và thứ tự lựa chọn feature ngẫu nhiên. Phiên bản `sklearn` nhanh hơn đáng kể nhờ tối ưu hóa nội bộ và hỗ trợ multi-threading.

### Hiệu năng thực sự của mô hình
So với Decision Tree đơn lẻ (R² ≈ 0.48), **Random Forest cải thiện đáng kể** nhờ:
* Giảm **Variance** bằng cách trung bình hóa nhiều cây.
* Bootstrap Sampling và Feature Randomness tạo ra sự đa dạng giữa các cây.
* Ít bị ảnh hưởng bởi outlier và noise hơn.

### Hướng giải quyết tiếp theo
Để tiếp tục cải thiện, có thể thử nghiệm **Gradient Boosting Regressor** — thay vì huấn luyện các cây song song và độc lập như Random Forest, Gradient Boosting huấn luyện **tuần tự**, mỗi cây mới tập trung vào việc sửa sai số của cây trước, từ đó giảm cả **Bias** lẫn **Variance**.


## Lưu model

In [11]:
MODEL_DIR = './models'
os.makedirs(MODEL_DIR, exist_ok=True)

# Lưu model from scratch
joblib.dump(rfr_grid_search.best_estimator_,
            os.path.join(MODEL_DIR, 'random_forest_scratch.pkl'))
print(f"Mô hình Random Forest From Scratch đã được lưu tại "
      f"{os.path.join(MODEL_DIR, 'random_forest_scratch.pkl')}")

# Lưu model sklearn
joblib.dump(sklearn_best,
            os.path.join(MODEL_DIR, 'random_forest_sklearn.pkl'))
print(f"Mô hình Random Forest sklearn đã được lưu tại "
      f"{os.path.join(MODEL_DIR, 'random_forest_sklearn.pkl')}")


Mô hình Random Forest From Scratch đã được lưu tại ./models\random_forest_scratch.pkl
Mô hình Random Forest sklearn đã được lưu tại ./models\random_forest_sklearn.pkl
